In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
import wandb
from huggingface_hub import HfApi, login
import matplotlib.pyplot as plt
import numpy as np
import time
from tqdm.auto import tqdm
import os
import pandas as pd
from IPython.display import display

# GPU Optimization check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cpu


In [3]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumarmayank_25afi14 (kumarmayank_25afi14-dtu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from google.colab import userdata
try:
    hf_token = userdata.get('')
    login(token=hf_token)
except:
    print("HF_TOKEN secret not found. Please login interactively:")
    login()

HF_TOKEN secret not found. Please login interactively:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
batch_size = 256
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

total_data = ConcatDataset([train_data, test_data])
total_len = len(total_data)
train_len = int(0.8 * total_len)
val_len = int(0.1 * total_len)
test_len = total_len - train_len - val_len

train_ds, val_ds, test_ds = random_split(
    total_data,
    [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 228kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.21MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.8MB/s]

Train: 56000 | Val: 7000 | Test: 7000


In [5]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return x_reconstructed

class VAE(nn.Module):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)

        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        return self.fc_mu(h3), self.fc_logvar(h3)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        x = x.view(x.size(0), -1)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_reconstructed = self.decoder(z)
        return x_reconstructed, mu, logvar

In [6]:
def calculate_loss(model_type, x_recon, x, loss_type="BCE", mu=None, logvar=None):
    x = x.view(x.size(0), -1)

    if loss_type == "BCE":
        recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')
    elif loss_type == "MSE":
        recon_loss = F.mse_loss(x_recon, x, reduction='sum')

    if model_type == "AE":
        return recon_loss / x.size(0)

    elif model_type == "VAE":
        # KL Divergence Loss
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        total_loss = (recon_loss + kld_loss) / x.size(0)
        return total_loss

In [7]:
def train_and_evaluate(model_type, latent_dim, loss_type, opt_name, epochs=10):
    run = wandb.init(
        project="dl-lab-8-ae-vae",
        name=f"{model_type}_{latent_dim}_{loss_type}_{opt_name}",
        config={"model": model_type, "latent_dim": latent_dim, "loss": loss_type, "optimizer": opt_name},
    )

    model = Autoencoder(latent_dim).to(device) if model_type == "AE" else VAE(latent_dim).to(device)

    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
    elif opt_name == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
    elif opt_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=1e-3)

    print(f"{m}_{dim}_{loss_fn}_{opt}")

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for data, _ in pbar:
            data = data.to(device)
            optimizer.zero_grad()

            if model_type == "AE":
                recon = model(data)
                loss = calculate_loss("AE", recon, data, loss_type)
            else:
                recon, mu, logvar = model(data)
                loss = calculate_loss("VAE", recon, data, loss_type, mu, logvar)

            loss.backward()
            train_loss += loss.item()
            optimizer.step()
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, _ in val_loader:
                data = data.to(device)
                if model_type == "AE":
                    recon = model(data)
                    loss = calculate_loss("AE", recon, data, loss_type)
                else:
                    recon, mu, logvar = model(data)
                    loss = calculate_loss("VAE", recon, data, loss_type, mu, logvar)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        epoch_time = time.time() - start_time

        wandb.log({"train_loss": avg_train_loss, "val_loss": avg_val_loss})

    reconstruct(model, model_type, test_loader)
    generate(model, model_type, latent_dim)
    interpolate(model, model_type, test_loader)

    run.finish()
    return model, {
        "Model": model_type,
        "Latent Dim": latent_dim,
        "Loss Fn": loss_type,
        "Optimizer": opt_name,
        "Train Loss": round(avg_train_loss, 4),
        "Validation Loss": round(avg_val_loss, 4),
    }

In [ ]:
def reconstruct(model, model_type, dataloader):
    model.eval()
    data, _ = next(iter(dataloader))
    data = data.to(device)[:8]
    with torch.no_grad():
        recon = model(data) if model_type == "AE" else model(data)[0]

    recon = recon.view(-1, 1, 28, 28).cpu()
    data = data.cpu()

    fig, axes = plt.subplots(2, 8, figsize=(15, 4))
    for i in range(8):
        axes[0, i].imshow(data[i].squeeze(), cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].imshow(recon[i].squeeze(), cmap='gray')
        axes[1, i].axis('off')

    wandb.log({"Reconstructions": wandb.Image(fig)})
    plt.close()

In [8]:
def generate(model, model_type, latent_dim, num_samples=8):
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, latent_dim).to(device)
        generated = model.decoder(z).view(-1, 1, 28, 28).cpu()

    fig, axes = plt.subplots(2, 4, figsize=(6, 4))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(generated[i].squeeze(), cmap='gray')
        ax.axis('off')

    wandb.log({"Generation": wandb.Image(fig)})
    plt.close()

In [9]:
def interpolate(model, model_type, dataloader, steps=10):
    model.eval()
    data, labels = next(iter(dataloader))
    data = data.to(device)
    labels = labels.to(device)

    idx1 = (labels == 7).nonzero(as_tuple=True)[0][0] #sneaker
    idx2 = (labels == 9).nonzero(as_tuple=True)[0][0] #ankle boot

    x1 = data[idx1:idx1+1]
    x2 = data[idx2:idx2+1]

    with torch.no_grad():
        if model_type == "AE":
            z1 = model.encoder(x1.view(1, -1))
            z2 = model.encoder(x2.view(1, -1))
        else:
            z1, _ = model.encode(x1.view(1, -1))
            z2, _ = model.encode(x2.view(1, -1))

    alphas = np.linspace(0, 1, steps)
    interpolated_images = []

    with torch.no_grad():
        for alpha in alphas:
            z_interp = (1 - alpha) * z1 + alpha * z2
            x_interp = model.decoder(z_interp)
            interpolated_images.append(x_interp.view(28, 28).cpu().numpy())

    fig, axes = plt.subplots(1, steps, figsize=(20, 2))
    for i, img in enumerate(interpolated_images):
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f"a={alphas[i]:.1f}")

    wandb.log({"Interpolation": wandb.Image(fig)})
    plt.close()

In [22]:
latent_dims = [16, 32]
models = ['AE', 'VAE']
losses = ['BCE', 'MSE']
optimizers = ['Adam', 'RMSprop', 'SGD']

trained_models = {}
results = []

for m in models:
    for dim in latent_dims:
        for loss_fn in losses:
            for opt in optimizers:
                model, metrics = train_and_evaluate(m, dim, loss_fn, opt, epochs=20)
                results.append(metrics)
                model_name = f"{m}_{dim}_{loss_fn}_{opt}"
                trained_models[model_name] = model

AE_16_BCE_Adam


Epoch 1/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 2/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 3/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 4/20:   0%|          | 0/219 [00:00<?, ?it/s]

...

Epoch 17/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 18/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 19/20:   0%|          | 0/219 [00:00<?, ?it/s]

Epoch 20/20:   0%|          | 0/219 [00:00<?, ?it/s]

train_loss,█▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val_loss,█▇▇▇▇▇▆▆▆▅▄▃▃▃▃▃▂▂▂▁
train_loss,49.00788
val_loss,48.12426


In [23]:
df_results = pd.DataFrame(results)
display(df_results)

,Model,Latent Dim,Loss Fn,Optimizer,Train Loss,Validation Loss
0,AE,16,BCE,Adam,218.5429,219.4113
1,AE,16,BCE,RMSprop,226.3346,227.2079
2,AE,16,BCE,SGD,255.9613,254.7261
3,AE,16,MSE,Adam,9.8753,10.3294
4,AE,16,MSE,RMSprop,12.0665,12.2537
5,AE,16,MSE,SGD,31.3128,31.5218
6,AE,32,BCE,Adam,217.7582,218.5139
7,AE,32,BCE,RMSprop,226.4739,229.8667
8,AE,32,BCE,SGD,257.5230,258.2300
9,AE,32,MSE,Adam,9.6060,9.7862


In [24]:
repo_id = "MK80310/DLLab8"

api = HfApi()

try:
    api.create_repo(repo_id=repo_id, exist_ok=True)

    for name, model in trained_models.items():
        save_path = f"{name}.pth"
        torch.save(model.state_dict(), save_path)

        api.upload_file(
            path_or_fileobj=save_path,
            path_in_repo=f"models/{save_path}",
            repo_id=repo_id,
            commit_message=f"Upload {name}"
        )

    print("Models successfully pushed to Hugging Face Hub!")

except Exception as e:
    print(f"Error during upload: {e}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  AE_16_BCE_Adam.pth          :  12%|#2        |  554kB / 4.56MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  VAE_32_MSE_RMSprop.pth      :  84%|########4 | 3.87MB / 4.59MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  VAE_32_MSE_SGD.pth          :  85%|########5 | 3.91MB / 4.59MB            

Models successfully pushed to Hugging Face Hub!


In [11]:
from huggingface_hub import hf_hub_download

repo_id = "MK80310/DLLab8"
latent_dims = [16, 32]
models = ['AE', 'VAE']
losses = ['BCE', 'MSE']
optimizers = ['Adam', 'RMSprop', 'SGD']

test_results = []

for m in models:
    for dim in latent_dims:
        for loss_fn in losses:
            for opt in optimizers:
                model_name = f"{m}_{dim}_{loss_fn}_{opt}"
                file_name = f"{model_name}.pth"
                try:
                    repo_file_path = f"models/{file_name}"
                    downloaded_model_path = hf_hub_download(repo_id=repo_id, filename=repo_file_path)
                    if m == "AE":
                        model = Autoencoder(latent_dim=dim).to(device)
                    else:
                        model = VAE(latent_dim=dim).to(device)
                    model.load_state_dict(torch.load(downloaded_model_path, map_location=device))
                    model.eval()

                    test_loss = 0
                    with torch.no_grad():
                        for data, _ in test_loader:
                            data = data.to(device)
                            if m == "AE":
                                recon = model(data)
                                loss = calculate_loss("AE", recon, data, loss_fn)
                            else:
                                recon, mu, logvar = model(data)
                                loss = calculate_loss("VAE", recon, data, loss_fn, mu, logvar)

                            test_loss += loss.item()

                    avg_test_loss = test_loss / len(test_loader)
                    test_results.append({
                        "Model Architecture": m,
                        "Latent Dimension": dim,
                        "Loss Function": loss_fn,
                        "Optimizer": opt,
                        "Test Loss": round(avg_test_loss, 4)
                    })

                except Exception as e:
                    print(f"❌ Failed to load or evaluate {model_name}: {e}")

models/AE_32_BCE_SGD.pth:   0%|          | 0.00/4.57M [00:00<?, ?B/s]

models/AE_32_MSE_Adam.pth:   0%|          | 0.00/4.57M [00:00<?, ?B/s]

models/AE_32_MSE_RMSprop.pth:   0%|          | 0.00/4.57M [00:00<?, ?B/s]

models/AE_32_MSE_SGD.pth:   0%|          | 0.00/4.57M [00:00<?, ?B/s]

models/VAE_16_BCE_Adam.pth:   0%|          | 0.00/4.56M [00:00<?, ?B/s]

...

models/VAE_32_BCE_SGD.pth:   0%|          | 0.00/4.59M [00:00<?, ?B/s]

models/VAE_32_MSE_Adam.pth:   0%|          | 0.00/4.59M [00:00<?, ?B/s]

models/VAE_32_MSE_RMSprop.pth:   0%|          | 0.00/4.59M [00:00<?, ?B/s]

models/VAE_32_MSE_SGD.pth:   0%|          | 0.00/4.59M [00:00<?, ?B/s]


FINAL TEST LOSS COMPARISON TABLE


In [12]:
df_test_results = pd.DataFrame(test_results)

display(df_test_results)

,Model Architecture,Latent Dimension,Loss Function,Optimizer,Test Loss
0,AE,16,BCE,Adam,219.9332
1,AE,16,BCE,RMSprop,227.7180
2,AE,16,BCE,SGD,255.6806
3,AE,16,MSE,Adam,10.2701
4,AE,16,MSE,RMSprop,12.1703
5,AE,16,MSE,SGD,31.8545
6,AE,32,BCE,Adam,219.1310
7,AE,32,BCE,RMSprop,230.4311
8,AE,32,BCE,SGD,259.0541
9,AE,32,MSE,Adam,9.7391
